In [36]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import os

base = "/content/project-root"

folders = [
    "src",
    "results",
    "checkpoints",
    "data",
    "notebooks"
]

os.makedirs(base, exist_ok=True)

for folder in folders:
    os.makedirs(
        os.path.join(base, folder),
        exist_ok=True
    )

print("PROJECT STRUCTURE CREATED")

PROJECT STRUCTURE CREATED


In [38]:
import os
import torch

os.environ['TORCH'] = torch.__version__

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-geometric
!pip install -q seaborn pandas matplotlib scikit-learn

In [39]:
model_code = r'''
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv


class BottleneckHybridGNN(torch.nn.Module):

    def __init__(self, num_features, num_classes):

        super(BottleneckHybridGNN, self).__init__()

        self.encoder = torch.nn.Linear(
            num_features,
            64
        )

        self.conv1 = GATConv(
            64,
            8,
            heads=8,
            dropout=0.6
        )

        self.conv2 = GATConv(
            64,
            num_classes,
            heads=1,
            concat=False,
            dropout=0.6
        )

        self.shortcut = torch.nn.Linear(
            num_features,
            num_classes
        )

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        x_encoded = F.elu(
            self.encoder(x)
        )

        x1 = F.elu(
            self.conv1(x_encoded, edge_index)
        )

        x1 = F.dropout(
            x1,
            p=0.6,
            training=self.training
        )

        x2 = self.conv2(
            x1,
            edge_index
        )

        return F.log_softmax(
            x2 + self.shortcut(x),
            dim=1
        )
'''

with open('/content/project-root/src/model.py', 'w') as f:
    f.write(model_code)

print("model.py created")

model.py created


In [40]:
dataset_code = r'''
from torch_geometric.datasets import Planetoid, Amazon
import torch_geometric.transforms as T
import torch
import numpy as np


def create_masks(data):

    num_nodes = data.num_nodes

    indices = np.random.permutation(num_nodes)

    train_size = int(0.6 * num_nodes)
    val_size = int(0.2 * num_nodes)

    train_idx = indices[:train_size]
    val_idx = indices[
        train_size:train_size + val_size
    ]
    test_idx = indices[
        train_size + val_size:
    ]

    data.train_mask = torch.zeros(
        num_nodes,
        dtype=torch.bool
    )

    data.val_mask = torch.zeros(
        num_nodes,
        dtype=torch.bool
    )

    data.test_mask = torch.zeros(
        num_nodes,
        dtype=torch.bool
    )

    data.train_mask[train_idx] = True
    data.val_mask[val_idx] = True
    data.test_mask[test_idx] = True

    return data


def get_data(name):

    path = './data/' + name

    if name in [
        'Cora',
        'CiteSeer',
        'PubMed'
    ]:

        dataset = Planetoid(
            root=path,
            name=name,
            transform=T.NormalizeFeatures()
        )

        data = dataset[0]

    else:

        dataset = Amazon(
            root=path,
            name='Computers',
            transform=T.NormalizeFeatures()
        )

        data = dataset[0]

        data = create_masks(data)

    return dataset, data
'''

with open('/content/project-root/src/dataset.py', 'w') as f:
    f.write(dataset_code)

print("dataset.py created")

dataset.py created


In [41]:
utils_code = r'''
def dummy():
    return True
'''

with open('/content/project-root/src/utils.py', 'w') as f:
    f.write(utils_code)

print("utils.py created")

utils.py created


In [42]:
train_code = r'''
import os
import copy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F

from sklearn.metrics import confusion_matrix

from src.model import BottleneckHybridGNN
from src.dataset import get_data


os.makedirs('results', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

datasets_list = [
    'Cora',
    'CiteSeer',
    'PubMed',
    'Computers'
]

all_results = {}

print("TRAINING STARTED")


for name in datasets_list:

    dataset, data = get_data(name)

    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    data = data.to(device)

    max_epochs = 400 if name in [
        'PubMed',
        'Computers'
    ] else 200

    run_accs = []

    print(f'\\nDATASET: {name}')

    best_model = None

    for run in range(1, 11):

        model = BottleneckHybridGNN(
            dataset.num_features,
            dataset.num_classes
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.01,
            weight_decay=5e-4
        )

        best_val_acc = 0
        final_test_acc = 0

        for epoch in range(1, max_epochs + 1):

            model.train()

            optimizer.zero_grad()

            out = model(data)

            loss = F.nll_loss(
                out[data.train_mask],
                data.y[data.train_mask]
            )

            loss.backward()

            optimizer.step()

            model.eval()

            with torch.no_grad():

                logits = model(data)

                val_acc = (
                    logits[data.val_mask].argmax(1)
                    ==
                    data.y[data.val_mask]
                ).sum().item() / data.val_mask.sum().item()

                test_acc = (
                    logits[data.test_mask].argmax(1)
                    ==
                    data.y[data.test_mask]
                ).sum().item() / data.test_mask.sum().item()

                if val_acc > best_val_acc:

                    best_val_acc = val_acc

                    final_test_acc = test_acc

                    best_model = copy.deepcopy(
                        model.state_dict()
                    )

        run_accs.append(final_test_acc)

        print(
            f'Run {run:02d} | '
            f'Test Accuracy: {final_test_acc:.4f}'
        )

    mean_score = np.mean(run_accs)
    std_score = np.std(run_accs)

    all_results[name] = {
        'mean_accuracy': float(mean_score),
        'std_accuracy': float(std_score)
    }

    torch.save(
        best_model,
        f'checkpoints/{name}_model_weights.pth'
    )

    with open(
        f'results/{name}_metrics.json',
        'w'
    ) as f:

        json.dump(
            all_results[name],
            f,
            indent=4
        )

    print(
        f'{name} FINAL: '
        f'{mean_score:.4f} ± {std_score:.4f}'
    )


names = list(all_results.keys())

means = [
    all_results[n]['mean_accuracy']
    for n in names
]

stds = [
    all_results[n]['std_accuracy']
    for n in names
]

plt.figure(figsize=(10, 6))

bars = plt.bar(
    names,
    means,
    yerr=stds,
    capsize=10
)

plt.title('Final Comparison Study')

plt.ylabel('Accuracy')

plt.savefig(
    'results/final_comparison_study.png'
)

plt.close()

summary_df = pd.DataFrame(all_results).T

summary_df.to_csv(
    'results/training_log.csv',
    index=False
)

print("\\nALL OUTPUTS SAVED")
'''

with open('/content/project-root/train.py', 'w') as f:
    f.write(train_code)

print("train.py created")

train.py created


In [43]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

from src.model import BottleneckHybridGNN
from src.dataset import get_data


os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)


datasets_list = [
    'Cora',
    'CiteSeer',
    'PubMed',
    'Computers'
]

all_metrics = []


for dataset_name in datasets_list:

    print(f'\nRUNNING INFERENCE FOR: {dataset_name}')

    # ============================================
    # LOAD DATA
    # ============================================

    dataset, data = get_data(dataset_name)

    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    data = data.to(device)

    # ============================================
    # LOAD MODEL
    # ============================================

    model = BottleneckHybridGNN(
        dataset.num_features,
        dataset.num_classes
    ).to(device)

    model.load_state_dict(
        torch.load(
            f'checkpoints/{dataset_name}_model_weights.pth',
            map_location=device
        )
    )

    # ============================================
    # INFERENCE
    # ============================================

    model.eval()

    with torch.no_grad():

        out = model(data)

        pred = out.argmax(dim=1)

    # ============================================
    # ACCURACY
    # ============================================

    test_acc = (

        pred[data.test_mask]
        ==
        data.y[data.test_mask]

    ).sum().item() / data.test_mask.sum().item()

    print(f'Test Accuracy: {test_acc:.4f}')

    all_metrics.append({

        'dataset': dataset_name,
        'test_accuracy': test_acc

    })

    # ============================================
    # CONFUSION MATRIX
    # ============================================

    y_pred = pred[data.test_mask].cpu().numpy()

    y_true = data.y[data.test_mask].cpu().numpy()

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False
    )

    plt.title(
        f'Confusion Matrix: {dataset_name}'
    )

    plt.xlabel('Predicted Label')

    plt.ylabel('True Label')

    plt.savefig(
        f'results/{dataset_name}_confusion_matrix.png'
    )

    plt.close()

    # ============================================
    # SAMPLE CSV FROM MODEL OUTPUT
    # ============================================

    sample_size = 10

    sample_df = pd.DataFrame({

        'node_id':
            list(range(sample_size)),

        'true_label':
            data.y[:sample_size].cpu().numpy(),

        'predicted_label':
            pred[:sample_size].cpu().numpy()

    })

    sample_df.to_csv(

        f'data/{dataset_name}_sample_predictions.csv',

        index=False
    )

    print(
        f'Sample predictions saved for {dataset_name}'
    )


# ============================================
# SAVE OVERALL METRICS
# ============================================

metrics_df = pd.DataFrame(all_metrics)

metrics_df.to_csv(
    'results/inference_metrics.csv',
    index=False
)

print('\nALL INFERENCE RESULTS SAVED SUCCESSFULLY')


RUNNING INFERENCE FOR: Cora
Test Accuracy: 0.8030
Sample predictions saved for Cora

RUNNING INFERENCE FOR: CiteSeer
Test Accuracy: 0.7030
Sample predictions saved for CiteSeer

RUNNING INFERENCE FOR: PubMed
Test Accuracy: 0.7700
Sample predictions saved for PubMed

RUNNING INFERENCE FOR: Computers
Test Accuracy: 0.8739
Sample predictions saved for Computers

ALL INFERENCE RESULTS SAVED SUCCESSFULLY


In [44]:
%cd /content/project-root

%run inference.py

/content/project-root
Test Accuracy: 0.8800


<Figure size 640x480 with 0 Axes>

In [45]:
requirements = """
torch==2.6.0
torch-geometric==2.6.1
torch-scatter
torch-sparse
numpy==2.0.2
pandas==2.2.2
matplotlib==3.10.0
seaborn==0.13.2
scikit-learn==1.6.1
"""

with open(
    '/content/project-root/requirements.txt',
    'w'
) as f:

    f.write(requirements)

print("requirements.txt created")



with open('/content/project-root/README.md', 'w') as f:
    f.write('# Hybrid GNN Research Project')

print("remaining files created")

requirements.txt created
remaining files created


In [46]:
config = """
model: BottleneckHybridGNN

datasets:
  - Cora
  - CiteSeer
  - PubMed
  - Computers

training:
  learning_rate: 0.01
  weight_decay: 0.0005
  dropout: 0.6
  epochs: 30
  runs: 2

optimizer:
  type: Adam

hardware:
  device: cuda_if_available

outputs:
  save_checkpoints: true
  save_metrics: true
  save_plots: true
"""

with open(
    '/content/project-root/config.yaml',
    'w'
) as f:

    f.write(config)

print("config.yaml created")

config.yaml created


In [47]:
%cd /content/project-root

%run train.py

/content/project-root
TRAINING STARTED
\nDATASET: Cora
Run 01 | Test Accuracy: 0.8070
Run 02 | Test Accuracy: 0.8080
Run 03 | Test Accuracy: 0.8250
Run 04 | Test Accuracy: 0.8050
Run 05 | Test Accuracy: 0.8040
Run 06 | Test Accuracy: 0.8000
Run 07 | Test Accuracy: 0.8110
Run 08 | Test Accuracy: 0.8130
Run 09 | Test Accuracy: 0.8310
Run 10 | Test Accuracy: 0.8000
Cora FINAL: 0.8104 ± 0.0097
\nDATASET: CiteSeer
Run 01 | Test Accuracy: 0.6870
Run 02 | Test Accuracy: 0.7130
Run 03 | Test Accuracy: 0.6820
Run 04 | Test Accuracy: 0.6850
Run 05 | Test Accuracy: 0.6970
Run 06 | Test Accuracy: 0.6910
Run 07 | Test Accuracy: 0.7020
Run 08 | Test Accuracy: 0.6890
Run 09 | Test Accuracy: 0.7040
Run 10 | Test Accuracy: 0.7060
CiteSeer FINAL: 0.6956 ± 0.0098
\nDATASET: PubMed
Run 01 | Test Accuracy: 0.7790
Run 02 | Test Accuracy: 0.7620
Run 03 | Test Accuracy: 0.7700
Run 04 | Test Accuracy: 0.7490
Run 05 | Test Accuracy: 0.7710
Run 06 | Test Accuracy: 0.7750
Run 07 | Test Accuracy: 0.7710
Run 08 | T

<Figure size 640x480 with 0 Axes>

In [48]:
import os

print(os.listdir('/content/project-root'))

['README.md', 'src', 'data', 'checkpoints', 'config.yaml', 'train.py', 'results', 'notebooks', 'inference.py', 'requirements.txt']


In [49]:
!cat /content/project-root/data/sample_data.csv

node_id,true_label,predicted_label
0,4,4
1,4,4
2,8,8
3,4,4
4,6,8
5,1,1
6,8,1
7,4,4
8,1,1
9,1,1


In [50]:
%cd /content

!zip -r project-root.zip project-root

/content
  adding: project-root/ (stored 0%)
  adding: project-root/README.md (stored 0%)
  adding: project-root/src/ (stored 0%)
  adding: project-root/src/dataset.py (deflated 67%)
  adding: project-root/src/__pycache__/ (stored 0%)
  adding: project-root/src/__pycache__/dataset.cpython-312.pyc (deflated 45%)
  adding: project-root/src/__pycache__/model.cpython-312.pyc (deflated 44%)
  adding: project-root/src/model.py (deflated 68%)
  adding: project-root/src/utils.py (stored 0%)
  adding: project-root/data/ (stored 0%)
  adding: project-root/data/CiteSeer_sample_predictions.csv (deflated 24%)
  adding: project-root/data/PubMed_sample_predictions.csv (deflated 28%)
  adding: project-root/data/Computers_sample_predictions.csv (deflated 25%)
  adding: project-root/data/Computers/ (stored 0%)
  adding: project-root/data/Computers/Computers/ (stored 0%)
  adding: project-root/data/Computers/Computers/raw/ (stored 0%)
  adding: project-root/data/Computers/Computers/raw/amazon_electronics

In [51]:
from google.colab import files

files.download('/content/project-root.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>